In [35]:
import os
from dotenv import load_dotenv
import mysql.connector
import pandas as pd

In [36]:
load_dotenv()

connection = mysql.connector.connect(
    host=os.getenv("DB_HOST"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME")
)

cursor = connection.cursor()

In [39]:
# Calculate the total orader value

query ="""
SELECT order_id,
SUM(price + freight_value) AS total_order_value
FROM order_items
GROUP BY order_id;
"""


df_order_value = pd.read_sql(query, connection)

df_order_value.head()

C:\Users\iseak\AppData\Local\Temp\ipykernel_4960\168829592.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_order_value = pd.read_sql(query, connection)


,order_id,total_order_value
0,00010242fe8c5a6d1ba2dd792cb16214,72.19
1,00018f77f2f0320c557190d7a144bdd3,259.83
2,000229ec398224ef6ca0657da4fc703e,216.87
3,00024acbcdf0a6daa1e931b038114c75,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04


In [ ]:
#Calculate delivery days for each order

query ="""
SELECT order_id,
DATEDIFF(order_delivered_customer_date, order_purchase_timestamp) AS delivery_day
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
"""

    
df_delivery_days = pd.read_sql(query, connection)

df_delivery_days.head()


In [ ]:
# Calculate delivery delay
query ="""
SELECT order_id,
DATEDIFF(order_delivered_customer_date, order_estimated_delivery_date) AS delivery_delay_days
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
AND order_estimated_delivery_date IS NOT NULL;
"""
    
df_delivery_delay = pd.read_sql(query, connection)

df_delivery_delay.head()

In [ ]:
# Calculate how many orders each customer has placed
query ="""
SELECT customer_id,
COUNT(order_id) AS customer_order_count
FROM orders
GROUP BY customer_id;
"""

df_customer_orders = pd.read_sql(query, connection)

df_customer_orders.head()

In [ ]:
# Calculates the total amount spent by each customer across all their orders

query ="""
SELECT o.customer_id,
SUM(oi.price + oi.freight_value) AS customer_total_spending
FROM orders o
JOIN order_items oi
ON o.order_id = oi.order_id
GROUP BY o.customer_id;
"""

df_total_spend = pd.read_sql(query, connection)

df_total_spend.head()

In [ ]:
# Calculates the average spending per order for each customer

query = """
SELECT o.customer_id,
AVG(order_totals.total_order_value) AS average_order_value
FROM orders o
JOIN(
    SELECT order_id,
    SUM(price + freight_value) AS total_order_value
    FROM order_items
    GROUP BY order_id
)AS order_totals
ON o.order_id = order_totals.order_id
GROUP BY o.customer_id;
"""

df_avg_spend = pd.read_sql(query, connection)

df_avg_spend.head()

In [ ]:
#Calculates the total revenue generated by each seller
query = """
SELECT seller_id,
SUM(price + freight_value) AS seller_revenue
FROM order_items
GROUP BY seller_id
ORDER BY seller_revenue DESC;
"""

df_total_revenue = pd.read_sql(query, connection)

df_total_revenue.head()

In [ ]:
# Calculates the number of orders handled by each seller.
query = """
SELECT seller_id,
COUNT(DISTINCT order_id) AS seller_order_count
FROM order_items
GROUP BY seller_id
ORDER BY seller_order_count DESC;
"""
df_total_orders = pd.read_sql(query, connection)

df_total_orders.head()

In [ ]:
# Identifies customers who have placed more than one order.

query ="""
SELECT customer_id,
COUNT(order_id) AS customer_order_count,
CASE 
WHEN COUNT(order_id) > 1 THEN 1
ELSE 0
END AS repeat_customer
FROM orders
GROUP BY customer_id;
"""
df_customer_total_order = pd.read_sql(query, connection)

df_customer_total_order.head()


In [ ]:
# Identifies repeat customers .

query = """
SELECT
    CASE
        WHEN customer_order_count > 1 THEN 'Repeat Customer'
        ELSE 'One-Time Customer'
    END AS customer_type,
    COUNT(*) AS total_customers
FROM (
    SELECT
        c.customer_unique_id,
        COUNT(o.order_id) AS customer_order_count
    FROM customers c
    JOIN orders o
        ON c.customer_id = o.customer_id
    GROUP BY c.customer_unique_id
) AS customer_orders
GROUP BY customer_type;
"""

df_repeat_customer = pd.read_sql(query,connection)

df_repeat_customer.head()

In [42]:
# On time delivery flag

query = """
SELECT
order_id,
order_delivered_customer_date,
order_estimated_delivery_date,
CASE
WHEN order_delivered_customer_date
<= order_estimated_delivery_date THEN 1
ELSE 0
END AS on_time_delivery
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
AND order_estimated_delivery_date IS NOT NULL;
"""

df_on_time_delivery = pd.read_sql(query,connection)

df_on_time_delivery.head()

C:\Users\iseak\AppData\Local\Temp\ipykernel_4960\4030488875.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_on_time_delivery = pd.read_sql(query,connection)


,order_id,order_delivered_customer_date,order_estimated_delivery_date,on_time_delivery
0,00010242fe8c5a6d1ba2dd792cb16214,2017-09-20 23:43:48,2017-09-29,1
1,00018f77f2f0320c557190d7a144bdd3,2017-05-12 16:04:24,2017-05-15,1
2,000229ec398224ef6ca0657da4fc703e,2018-01-22 13:19:16,2018-02-05,1
3,00024acbcdf0a6daa1e931b038114c75,2018-08-14 13:32:39,2018-08-20,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,2017-03-01 16:42:31,2017-03-17,1


In [ ]:
# Comparing on time delivery with purchase month and year

query = """
SELECT order_id,
MONTH(order_purchase_timestamp) AS purchase_month,
YEAR(order_purchase_timestamp) AS purchase_year
FROM orders
WHERE order_purchase_timestamp IS NOT NULL;
"""

df_on_time_delivery = pd.read_sql(query,connection)

df_on_time_delivery.head()

C:\Users\iseak\AppData\Local\Temp\ipykernel_4960\1845893794.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_ = pd.read_sql(query,connection)


,order_id,order_delivered_customer_date,order_estimated_delivery_date,on_time_delivery
0,00010242fe8c5a6d1ba2dd792cb16214,2017-09-20 23:43:48,2017-09-29,1
1,00018f77f2f0320c557190d7a144bdd3,2017-05-12 16:04:24,2017-05-15,1
2,000229ec398224ef6ca0657da4fc703e,2018-01-22 13:19:16,2018-02-05,1
3,00024acbcdf0a6daa1e931b038114c75,2018-08-14 13:32:39,2018-08-20,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,2017-03-01 16:42:31,2017-03-17,1


In [45]:
# Purchase day of week

query = """
SELECT order_id,
DAYNAME(order_purchase_timestamp) AS purchase_day,
CASE 
WHEN DAYOFWEEK(order_purchase_timestamp) IN (1,7)
THEN 'Weekend'
ELSE 'Weekday'
END AS purchase_day_type
FROM orders
WHERE order_purchase_timestamp IS NOT NULL;
"""

df_purchase_day = pd.read_sql(query,connection)

df_purchase_day.head()


C:\Users\iseak\AppData\Local\Temp\ipykernel_4960\3410392121.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_purchase_day = pd.read_sql(query,connection)


,order_id,purchase_day,purchase_day_type
0,00010242fe8c5a6d1ba2dd792cb16214,Wednesday,Weekday
1,00018f77f2f0320c557190d7a144bdd3,Wednesday,Weekday
2,000229ec398224ef6ca0657da4fc703e,Sunday,Weekend
3,00024acbcdf0a6daa1e931b038114c75,Wednesday,Weekday
4,00042b26cf59d7ce69dfabb4e55b4fd9,Saturday,Weekend


In [ ]:
# Freight cost percantage

query = """
SELECT order_id,
SUM(price) AS product_value,
SUM(freight_value) AS total_freight ,
ROUND(SUM(freight_value) / NULLIF(SUM(price + freight_value) ,0) * 100,
2) AS freight_percentage
FROM order_items
GROUP BY order_id;
"""

df_freight_cost = pd.read_sql(query,connection)

df_freight_cost.head()


C:\Users\iseak\AppData\Local\Temp\ipykernel_4960\1510635638.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_freight_cost = pd.read_sql(query,connection)


,order_id,product_value,total_freight,freight_percentage
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,18.41
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,7.67
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,8.24
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,49.61
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,8.32


In [ ]:
# Identifies which product categories generate the most revenue.

query = """
SELECT COALECSE(pct.product_category_name_english ,
p.product_category_name ,
'Unknown') AS product_category
ROUND(SUM(oi.price + oi.freight_value), 2) AS category_revenue
FROM order_item oi
JOIN products p
ON oi.product_id = p.product_id
LEFT JOIN product_category_translation pct
ON p.product_category_name = pct.product_category_name
GROUP BY product_category
ORDER BY category_revenue DESC
"""

df_revenue = pd.read_sql(query,connection)

df_revenue.head()

In [ ]:
cursor.close()
connection.close()